# Section 5: Closed-Loop VLM Simulation

**Paper Figure Generation for Closed-Loop Experiments**

This notebook produces publication-ready figures comparing 9 closed-loop control policies:

| Policy | Description |
|--------|-------------|
| `local_only` | Noisy planner argmax (baseline, no VLM) |
| `vlm_hold` | **Directly execute** the delayed VLM-selected trajectory (hold: max inflight = 1) |
| `vlm_stream` | **Directly execute** the delayed VLM-selected trajectory (stream: fixed-cadence querying; adopt newest-by-send time) |
| `vlm_hold_match` | Execute the **current** planner candidate closest to the delayed VLM trajectory (hold querying) |
| `vlm_stream_match` | Same as above, but stream-query (fixed-cadence querying) |
| `score_fusion` | Fuse planner score + VLM similarity (hold querying) |
| `prob_fusion` | Fuse in probability space (hold querying) |
| `score_fusion_stream` | Score fusion + stream-query (fixed cadence + pipelined by default) |
| `prob_fusion_stream` | Prob fusion + stream-query (fixed cadence + pipelined by default) |

---

**Recommended workflow (matches `scripts/closed_loop_sim/section5_closed_loop_experiments.sh`)**:
1. Run launcher step `1` (fixed-delay λ sweep; stored under `step3_lambda_sweep_fusions`)
2. Run launcher step `2` (main 0–5 s delay sweep; stored under `step4_main_results`)

Then run this notebook top-to-bottom (Sections 1 → 12).

In [ ]:
# =============================================================================
# Setup and Configuration
# =============================================================================

from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Try to import seaborn for nicer plots
try:
    import seaborn as sns

    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    sns = None
    print("seaborn not available, using matplotlib defaults")


# -----------------------------------------------------------------------------
# Paper-ready plot style (match `task2_paper_figures.ipynb`)
# -----------------------------------------------------------------------------

plt.rcParams.update(
    {
        "font.size": 12,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Liberation Sans"],
        "axes.labelsize": 14,
        "axes.titlesize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "figure.figsize": (8, 5),
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

if HAS_SEABORN:
    sns.set_theme(
        context="paper",
        style="whitegrid",
        palette="colorblind",
        font="sans-serif",
        rc={
            "axes.spines.right": False,
            "axes.spines.top": False,
            "axes.grid": True,
            "grid.alpha": 0.25,
            "grid.linestyle": "-",
            "legend.frameon": False,
            "legend.loc": "best",
            "lines.linewidth": 2,
            "axes.edgecolor": "#333333",
        },
    )
    sns.set_context("paper", font_scale=1)


# Consistent colorblind palette
if HAS_SEABORN:
    _PALETTE = sns.color_palette("colorblind", 12).as_hex()
else:
    # matplotlib defaults fallback (approx. tab10)
    _PALETTE = [
        "#1f77b4",
        "#ff7f0e",
        "#2ca02c",
        "#d62728",
        "#9467bd",
        "#8c564b",
        "#e377c2",
        "#7f7f7f",
        "#bcbd22",
        "#17becf",
        "#1f77b4",
        "#ff7f0e",
    ]


# Color palette for policies
POLICY_COLORS = {
    "local_only": _PALETTE[5],  # baseline-ish
    "vlm_hold": _PALETTE[4],
    "vlm_stream": _PALETTE[2],
    "vlm_hold_match": _PALETTE[6],
    "vlm_stream_match": _PALETTE[9],
    "score_fusion": _PALETTE[0],
    "prob_fusion": _PALETTE[3],
    # Stream-fusion variants: give them lighter shades for visual grouping.
    "score_fusion_stream": "#aec7e8",
    "prob_fusion_stream": "#c5b0d5",
}

POLICY_LABELS = {
    "local_only": "Local Only",
    "vlm_hold": "VLM Hold (direct)",
    "vlm_stream": "VLM Stream (direct)",
    "vlm_hold_match": "VLM Hold (match plan)",
    "vlm_stream_match": "VLM Stream (match plan)",
    "score_fusion": "Score Fusion (hold)",
    "prob_fusion": "Prob Fusion (hold)",
    "score_fusion_stream": "Score Fusion (stream)",
    "prob_fusion_stream": "Prob Fusion (stream)",
}

POLICY_MARKERS = {
    "local_only": "o",
    "vlm_hold": "s",
    "vlm_stream": "^",
    "vlm_hold_match": "X",
    "vlm_stream_match": "*",
    "score_fusion": "D",
    "prob_fusion": "p",
    "score_fusion_stream": "d",
    "prob_fusion_stream": "P",
}

POLICY_ORDER = [
    "local_only",
    "vlm_hold",
    "vlm_stream",
    "vlm_hold_match",
    "vlm_stream_match",
    "score_fusion",
    "prob_fusion",
    "score_fusion_stream",
    "prob_fusion_stream",
]


# Rounded-rectangle bars (paper figures style)
ROUNDED_BAR_RADIUS_PT = 6

# Global plotting toggle: error bars can clutter these figures.
PLOT_ERROR_BARS = False


def apply_rounded_bars(ax, bars, *, radius_pt: float = ROUNDED_BAR_RADIUS_PT):
    """Replace rectangular bars with rounded-rectangle PathPatches."""
    from matplotlib.patches import PathPatch
    from matplotlib.path import Path

    fig = ax.figure
    dpi = fig.dpi
    trans = ax.transData

    # Bezier approximation constant for a quarter circle.
    k = 0.5522847498307936

    def rounded_rect_path(x: float, y: float, w: float, h: float, rx: float, ry: float) -> Path:
        rx = max(0.0, min(rx, w * 0.5))
        ry = max(0.0, min(ry, h * 0.5))
        if rx == 0.0 or ry == 0.0:
            verts = [(x, y), (x + w, y), (x + w, y + h), (x, y + h), (x, y)]
            codes = [Path.MOVETO, Path.LINETO, Path.LINETO, Path.LINETO, Path.CLOSEPOLY]
            return Path(verts, codes)

        verts = [
            (x + rx, y),
            (x + w - rx, y),
            (x + w - rx + k * rx, y),
            (x + w, y + ry - k * ry),
            (x + w, y + ry),
            (x + w, y + h - ry),
            (x + w, y + h - ry + k * ry),
            (x + w - rx + k * rx, y + h),
            (x + w - rx, y + h),
            (x + rx, y + h),
            (x + rx - k * rx, y + h),
            (x, y + h - ry + k * ry),
            (x, y + h - ry),
            (x, y + ry),
            (x, y + ry - k * ry),
            (x + rx - k * rx, y),
            (x + rx, y),
        ]
        codes = [
            Path.MOVETO,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
            Path.LINETO,
            Path.CURVE4,
            Path.CURVE4,
            Path.CURVE4,
        ]
        # Close the path explicitly for consistent fills.
        verts.append((x + rx, y))
        codes.append(Path.CLOSEPOLY)
        return Path(verts, codes)

    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    for bar in list(bars):
        x = bar.get_x()
        y = bar.get_y()
        w = bar.get_width()
        h = bar.get_height()
        if w == 0 or h == 0:
            continue

        if h < 0:
            y = y + h
            h = -h

        # Desired radius in pixels.
        r_px = float(radius_pt) * dpi / 72.0

        # Convert to data-units in x/y using local axis scales.
        x0_px, y0_px = trans.transform((x, y))
        x1_px, _ = trans.transform((x + 1.0, y))
        _, y1_px = trans.transform((x, y + 1.0))
        sx = abs(x1_px - x0_px)
        sy = abs(y1_px - y0_px)
        rx = (r_px / sx) if sx else 0.0
        ry = (r_px / sy) if sy else 0.0

        # Clamp radii to the bar size.
        eps = 1e-12
        rx = max(0.0, min(rx, (w * 0.5) - eps))
        ry = max(0.0, min(ry, (h * 0.5) - eps))

        fc = bar.get_facecolor()
        ec = bar.get_edgecolor()
        lw = bar.get_linewidth()
        alpha = bar.get_alpha()
        z = bar.get_zorder()

        bar.remove()

        path = rounded_rect_path(x, y, w, h, rx, ry)
        patch = PathPatch(
            path,
            facecolor=fc,
            edgecolor=ec,
            linewidth=lw,
            alpha=alpha,
            transform=trans,
            zorder=z,
        )
        patch.set_clip_path(ax.patch)
        ax.add_patch(patch)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


# Automatically round any bars created via ax.bar
# NOTE: this cell may be executed multiple times in notebooks; make the monkey-patch idempotent
# (and attempt a best-effort recovery if it was double-patched previously).
try:
    import importlib

    import matplotlib.axes as _maxes
    import matplotlib.axes._axes as _axes_mod

    # Recovery: if we already patched `Axes.bar` but didn't preserve the original,
    # a second execution can set the "original" to the wrapper and recurse.
    if _maxes.Axes.bar.__name__ == "_bar_with_rounding" and not hasattr(
        _maxes.Axes, "_orig_bar_for_rounding"
    ):
        importlib.reload(_axes_mod)
        _maxes.Axes.bar = _axes_mod.Axes.bar

    if not getattr(_maxes.Axes, "_rounded_bar_patch_applied", False):
        _maxes.Axes._orig_bar_for_rounding = _maxes.Axes.bar

        def _bar_with_rounding(self, *args, **kwargs):
            bars = _maxes.Axes._orig_bar_for_rounding(self, *args, **kwargs)
            try:
                apply_rounded_bars(self, bars)
            except Exception:
                pass
            return bars

        _maxes.Axes.bar = _bar_with_rounding
        _maxes.Axes._rounded_bar_patch_applied = True
except Exception:
    pass


print("Setup complete!")

In [ ]:
# =============================================================================
# Data Loading Utilities (shared + notebook-friendly wrappers)
# =============================================================================
# This notebook is used to compare multiple experiment steps (delay sweep, query-rate sweep,
# lambda sweep, main results). We keep the data loading logic in a shared helper module so:
# - folder discovery matches `scripts/closed_loop_sim/section5_closed_loop_experiments.sh`
# - `vlm_query_hz` and effective request-Hz are always present
# - plots can be sorted in a consistent experiment order

import sys

# Ensure repo root is importable (so `import slow_brain_fast_planner...` works in notebooks)
_REPO_ROOT = Path("..").resolve()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from slow_brain_fast_planner.benchmarks.closed_loop_results import (  # noqa: E402
    load_closed_loop_run,
    load_section5_results_table,
    resolve_section5_root,
)


def load_run(run_dir: Path | str) -> dict:
    """Backward-compatible wrapper used by older cells in this notebook."""
    run_dir = Path(run_dir).expanduser().resolve()

    # Prefer row-level results for most plots
    df, cfg = load_closed_loop_run(run_dir, prefer="results")

    # Also load summary.csv if it exists (some sections use it)
    summary_path = run_dir / "summary.csv"
    df_summary = pd.read_csv(summary_path) if summary_path.exists() else None

    return {
        "run_dir": run_dir,
        "config": cfg,
        "df": df,
        "df_summary": df_summary,
    }


def aggregate_by_policy(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate results by policy and key parameters."""
    group_cols = ["policy", "delay_s", "lambda_sim", "staleness_tau_s", "similarity_mode"]
    group_cols = [c for c in group_cols if c in df.columns]

    metrics = ["mean_cte_m", "p95_cte_m", "chosen_switches", "mean_speed_mps"]
    metrics = [m for m in metrics if m in df.columns]

    agg = df.groupby(group_cols)[metrics].agg(["mean", "std", "count"]).reset_index()
    agg.columns = ["_".join(col).strip("_") for col in agg.columns]
    return agg


def get_best_config_per_policy(df: pd.DataFrame, metric: str = "mean_cte_m") -> pd.DataFrame:
    """Find best configuration for each policy."""
    metric_col = f"{metric}_mean" if f"{metric}_mean" in df.columns else metric

    best_rows = []
    for policy in df["policy"].unique():
        policy_df = df[df["policy"] == policy]
        if len(policy_df) == 0:
            continue
        best_idx = policy_df[metric_col].idxmin()
        best_rows.append(policy_df.loc[best_idx])

    return pd.DataFrame(best_rows).reset_index(drop=True)


print("Data loading utilities defined (slow_brain_fast_planner.benchmarks.closed_loop_results).")

In [ ]:
# =============================================================================
# Paper figure: delay curves under different planner difficulty
# =============================================================================
# Idea: show that VLM fusion helps when the planner is *meaningfully imperfect*.
#
# We generate three experiment folders (easy / medium / hard planner). Each folder should contain
# `step4_main_results/delay_*/.../summary.csv` produced by:
#   OUT_BASE=<folder> EPSILON=<...> NOISE_STD=<...> VLM_MISTAKE_PROB=<...> \
#     bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh all
#
# Then this cell plots delay->metric curves with one panel per difficulty.

from pathlib import Path

import matplotlib.pyplot as plt


def _load_step4_summary_curves(section5_root: Path) -> pd.DataFrame:
    step4 = section5_root / "step4_main_results"
    if not step4.exists():
        raise FileNotFoundError(f"Missing step4_main_results at: {step4}")

    def parse_delay(name: str):
        if not name.startswith("delay_"):
            return None
        try:
            return float(name.split("delay_", 1)[1])
        except Exception:
            return None

    def latest_run_dir(group_dir: Path):
        runs = sorted([p for p in group_dir.glob("closed_loop_fusion_*") if p.is_dir()])
        return runs[-1] if runs else None

    group_dirs = [
        "baselines",
        "score_fusion",
        "prob_fusion",
        "score_fusion_stream",
        "prob_fusion_stream",
    ]

    rows = []
    for delay_dir in sorted(
        [p for p in step4.glob("delay_*") if p.is_dir()],
        key=lambda p: parse_delay(p.name) or 1e9,
    ):
        delay_s = parse_delay(delay_dir.name)
        for group in group_dirs:
            gdir = delay_dir / group
            if not gdir.exists():
                continue
            run_dir = latest_run_dir(gdir)
            if run_dir is None:
                continue
            sp = run_dir / "summary.csv"
            if not sp.exists():
                continue
            df = pd.read_csv(sp)
            if "delay_s" not in df.columns and delay_s is not None:
                df["delay_s"] = delay_s
            df["delay_s"] = pd.to_numeric(df["delay_s"], errors="coerce")
            df["run_dir"] = str(run_dir)
            rows.append(df)

    if not rows:
        raise RuntimeError(f"No summary.csv found under: {step4}")

    df_all = pd.concat(rows, ignore_index=True)

    policies = [
        "planner_oracle",
        "local_only",
        "vlm_hold",
        "vlm_stream",
        "vlm_hold_match",
        "vlm_stream_match",
        "score_fusion",
        "prob_fusion",
        "score_fusion_stream",
        "prob_fusion_stream",
    ]
    df_all = df_all[df_all["policy"].isin(policies)].copy()

    # If multiple runs exist per (policy, delay), keep the newest by run_dir name.
    df_all = (
        df_all.sort_values(["delay_s", "policy", "run_dir"])
        .drop_duplicates(["delay_s", "policy"], keep="last")
        .sort_values(["delay_s", "policy"])
    )

    return df_all


SECTION5_ROOT = Path(str(globals().get("SECTION5_ROOT", "logs/section5_closed_loop"))).resolve()

# Three planner difficulty regimes.
# If these folders don’t exist yet, the cell will print the exact command to generate them.
REGIMES = [
    {
        "name": "easy planner",
        "out_base": Path(str(SECTION5_ROOT)).parent / "section5_closed_loop_easy",
        "env": {"EPSILON": "0.1", "NOISE_STD": "0.5", "VLM_MISTAKE_PROB": "0.0"},
    },
    {
        "name": "medium planner",
        "out_base": Path(str(SECTION5_ROOT)).parent / "section5_closed_loop_medium",
        "env": {"EPSILON": "0.3", "NOISE_STD": "1.0", "VLM_MISTAKE_PROB": "0.0"},
    },
    {
        "name": "hard planner",
        "out_base": Path(str(SECTION5_ROOT)).parent / "section5_closed_loop_hard",
        "env": {"EPSILON": "0.6", "NOISE_STD": "2.0", "VLM_MISTAKE_PROB": "0.1"},
    },
]

METRIC = "success_rate"  # try: mean_route_completion, mean_goal_dist_min_m, mean_cte_m
METRIC_TITLE = {
    "success_rate": "Success rate",
    "mean_route_completion": "Route completion",
    "mean_goal_dist_min_m": "Min goal dist (m)",
    "mean_cte_m": "Mean CTE (m)",
}.get(METRIC, METRIC)

policies = [p for p in (globals().get("POLICY_ORDER", None) or []) if p is not None]
if not policies:
    policies = [
        "planner_oracle",
        "local_only",
        "vlm_hold",
        "vlm_stream",
        "vlm_hold_match",
        "vlm_stream_match",
        "score_fusion",
        "prob_fusion",
        "score_fusion_stream",
        "prob_fusion_stream",
    ]

fig, axes = plt.subplots(1, len(REGIMES), figsize=(6.5 * len(REGIMES), 4.6), sharey=True)
if len(REGIMES) == 1:
    axes = [axes]

for ax, reg in zip(axes, REGIMES, strict=False):
    root = Path(reg["out_base"]).resolve()

    if not root.exists():
        env = " ".join([f"{k}={v}" for k, v in reg["env"].items()])
        print(f"[MISSING] {reg['name']}: {root}")
        print("  To generate:")
        print(
            f"  OUT_BASE={root} {env} WRITE_EXAMPLES=0 "
            "bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh all"
        )
        ax.set_title(reg["name"] + " (missing logs)")
        ax.set_xlabel("Delay (s)")
        ax.grid(True, alpha=0.25)
        continue

    df = _load_step4_summary_curves(root)
    delays = sorted(df["delay_s"].dropna().unique().tolist())

    for pol in [p for p in policies if p in set(df["policy"])]:
        sub = df[df["policy"] == pol].sort_values("delay_s")
        label = POLICY_LABELS.get(pol, pol) if "POLICY_LABELS" in globals() else pol
        color = POLICY_COLORS.get(pol, None) if "POLICY_COLORS" in globals() else None
        marker = POLICY_MARKERS.get(pol, "o") if "POLICY_MARKERS" in globals() else "o"
        ax.plot(
            sub["delay_s"],
            sub[METRIC],
            label=label,
            color=color,
            marker=marker,
            linewidth=2.0,
        )

    ax.set_title(f"{reg['name']}\n({METRIC_TITLE})")
    ax.set_xlabel("Delay (s)")
    ax.set_xticks(delays)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel(METRIC_TITLE)
axes[-1].legend(ncol=1, frameon=False, fontsize=9, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Figure (paper-ready): Success rate vs delay (split into readable panels)
# =============================================================================

from pathlib import Path

import matplotlib.pyplot as plt


def _best_by_policy_delay(df_src: pd.DataFrame) -> pd.DataFrame:
    required = ["policy", "delay_s", "success_rate"]
    missing = [c for c in required if c not in df_src.columns]
    if missing:
        raise KeyError(f"Missing columns in df_src: {missing}")

    dfp = df_src.copy()
    dfp["delay_s"] = pd.to_numeric(dfp["delay_s"], errors="coerce")

    # Pick best row per (policy, delay). Tie-break using RC/goal-dist/CTE if available.
    sort_cols = [
        "success_rate",
        ("mean_route_completion" if "mean_route_completion" in dfp.columns else None),
        (
            "mean_goal_dist_min_m"
            if "mean_goal_dist_min_m" in dfp.columns
            else ("mean_goal_dist_final_m" if "mean_goal_dist_final_m" in dfp.columns else None)
        ),
        ("mean_cte_m" if "mean_cte_m" in dfp.columns else None),
    ]
    sort_cols = [c for c in sort_cols if c is not None]

    asc_map = {
        "success_rate": False,
        "mean_route_completion": False,
        "mean_goal_dist_final_m": True,
        "mean_goal_dist_min_m": True,
        "mean_cte_m": True,
    }
    asc = [asc_map.get(c, True) for c in sort_cols]

    df_ranked = dfp.sort_values(sort_cols + ["policy", "delay_s"], ascending=asc + [True, True])
    df_best = df_ranked.drop_duplicates(["policy", "delay_s"], keep="first").sort_values(
        ["delay_s", "policy"]
    )
    return df_best


# Source preference:
# 1) if you already built df_best_delay_policy in another cell
# 2) else use df_all from the quick loader cell
# 3) else fall back to df_step4_summary/df_all_summary if present
# 4) else auto-scan SECTION5_ROOT/step4_main_results/*/summary.csv (so this cell can run standalone)


def _pick_section5_root() -> Path:
    candidates = []
    if "SECTION5_ROOT" in globals() and globals()["SECTION5_ROOT"] is not None:
        candidates.append(Path(str(globals()["SECTION5_ROOT"])))
    candidates += [
        Path("logs/section5_closed_loop"),
        Path("logs/section5_closed_loop"),
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(
        "Could not find SECTION5_ROOT. Tried: " + ", ".join(str(p) for p in candidates)
    )


def _parse_delay_from_dirname(name: str) -> float | None:
    if not name.startswith("delay_"):
        return None
    try:
        return float(name.split("delay_", 1)[1])
    except Exception:
        return None


def _latest_run_dir(group_dir: Path) -> Path | None:
    runs = sorted([p for p in group_dir.glob("closed_loop_fusion_*") if p.is_dir()])
    return runs[-1] if runs else None


def _load_summary_from_fs(section5_root: Path) -> pd.DataFrame:
    step4 = section5_root / "step4_main_results"
    if not step4.exists():
        raise FileNotFoundError(f"Missing step4_main_results at: {step4}")

    groups = [
        "baselines",
        "score_fusion",
        "prob_fusion",
        "score_fusion_stream",
        "prob_fusion_stream",
    ]

    rows = []
    delay_dirs = [p for p in step4.glob("delay_*") if p.is_dir()]
    for delay_dir in sorted(delay_dirs, key=lambda p: _parse_delay_from_dirname(p.name) or 1e9):
        delay_s = _parse_delay_from_dirname(delay_dir.name)
        for group in groups:
            gdir = delay_dir / group
            if not gdir.exists():
                continue
            run_dir = _latest_run_dir(gdir)
            if run_dir is None:
                continue
            sp = run_dir / "summary.csv"
            if not sp.exists():
                continue
            df = pd.read_csv(sp)
            if "delay_s" not in df.columns and delay_s is not None:
                df["delay_s"] = delay_s
            df["delay_s"] = pd.to_numeric(df["delay_s"], errors="coerce")
            df["run_dir"] = str(run_dir)
            rows.append(df)

    if not rows:
        raise RuntimeError(
            f"No summary.csv found under: {step4}. "
            "Run the closed-loop jobs first, "
            "e.g. `bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh all`."
        )

    return pd.concat(rows, ignore_index=True)


# These names are injected at runtime by the "Load Results" cells below (this cell
# supports out-of-order execution), so look them up via globals().
_df_best = globals().get("df_best_delay_policy")
_df_all_g = globals().get("df_all")
_df_step4_summary_g = globals().get("df_step4_summary")
_df_all_summary_g = globals().get("df_all_summary")

_df_src = None
if _df_best is not None and len(_df_best) > 0:
    df_best = _df_best.copy()
else:
    if _df_all_g is not None and len(_df_all_g) > 0:
        _df_src = _df_all_g.copy()
    elif _df_step4_summary_g is not None and len(_df_step4_summary_g) > 0:
        _df_src = _df_step4_summary_g.copy()
    elif _df_all_summary_g is not None and len(_df_all_summary_g) > 0:
        _df_src = _df_all_summary_g.copy()

    if _df_src is None or len(_df_src) == 0:
        _root = _pick_section5_root()
        _df_src = _load_summary_from_fs(_root)
        print(
            f"[auto-load] loaded {len(_df_src)} summary rows "
            f"from {_root}/step4_main_results (filesystem scan)"
        )

    if "delay_s" not in _df_src.columns and "section5_delay_sweep" in _df_src.columns:
        _df_src = _df_src.copy()
        _df_src["delay_s"] = _df_src["section5_delay_sweep"]

    df_best = _best_by_policy_delay(_df_src)

# ---- Discard delay > 5 ----
df_best = df_best[df_best["delay_s"] <= 5].copy()

# Policy sets
BASELINES = ["planner_oracle", "local_only"]
VLM_AND_MATCH = ["vlm_hold", "vlm_stream", "vlm_hold_match", "vlm_stream_match"]
FUSION = ["score_fusion", "prob_fusion", "score_fusion_stream", "prob_fusion_stream"]

FINAL_ALL_IN_ONE = [
    "planner_oracle",
    "local_only",
    "vlm_hold",
    "vlm_stream",
    "score_fusion_stream",
    "prob_fusion_stream",
]


# Only plot policies present in df_best
present = set(df_best["policy"].unique().tolist())
BASELINES = [p for p in BASELINES if p in present]
VLM_AND_MATCH = [p for p in VLM_AND_MATCH if p in present]
FUSION = [p for p in FUSION if p in present]

all_delays = sorted(pd.to_numeric(df_best["delay_s"], errors="coerce").dropna().unique().tolist())


def _plot_sr(ax, policies, title):
    for pol in policies:
        sub = df_best[df_best["policy"] == pol].sort_values("delay_s")
        if len(sub) == 0:
            continue
        label = POLICY_LABELS.get(pol, pol) if "POLICY_LABELS" in globals() else pol
        color = POLICY_COLORS.get(pol, None) if "POLICY_COLORS" in globals() else None
        marker = POLICY_MARKERS.get(pol, "o") if "POLICY_MARKERS" in globals() else "o"
        ax.plot(
            sub["delay_s"],
            sub["success_rate"],
            marker=marker,
            linewidth=2.0,
            label=label,
            color=color,
        )

    ax.set_title(title)
    ax.set_xlabel("Delay (s)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_xticks(all_delays)
    ax.grid(True, alpha=0.28)


# 1) Baselines
# if BASELINES:
#     fig, ax = plt.subplots(figsize=(7.6, 4.6))
#     _plot_sr(ax, BASELINES, "Success rate vs delay (baselines)")
#     ax.set_ylabel("Success rate")
#     ax.legend(frameon=False, fontsize=9)
#     fig.tight_layout()
#     if "PAPER_DIR" in globals():
#         out = (PAPER_DIR / "success_rate_vs_delay_baselines").resolve()
#         fig.savefig(str(out) + ".png", dpi=220)
#         fig.savefig(str(out) + ".pdf")
#         print("Saved:", str(out) + ".png/.pdf")
#     plt.show()

# 2) VLM direct + match
if VLM_AND_MATCH:
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    _plot_sr(ax, VLM_AND_MATCH, "Success rate vs delay (VLM direct + match)")
    ax.set_ylabel("Success rate")
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    _paper_dir = globals().get("PAPER_DIR")
    if _paper_dir is not None:
        out = (_paper_dir / "success_rate_vs_delay_vlm_and_match").resolve()
        fig.savefig(str(out) + ".png", dpi=220)
        fig.savefig(str(out) + ".pdf")
        print("Saved:", str(out) + ".png/.pdf")
    plt.show()

# 3) Fusion (the main story)
if FUSION:
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    _plot_sr(ax, FUSION, "Success rate vs delay (fusion)")
    ax.set_ylabel("Success rate")
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    _paper_dir = globals().get("PAPER_DIR")
    if _paper_dir is not None:
        out = (_paper_dir / "success_rate_vs_delay_fusion").resolve()
        fig.savefig(str(out) + ".png", dpi=220)
        fig.savefig(str(out) + ".pdf")
        print("Saved:", str(out) + ".png/.pdf")
    plt.show()

# Optional: one compact figure with all methods (for quick debugging)
if BASELINES and VLM_AND_MATCH and FUSION:
    fig, ax = plt.subplots(figsize=(6, 3))
    _plot_sr(ax, FINAL_ALL_IN_ONE, "Success rate vs delay (all methods)")
    ax.set_ylabel("Success rate")
    # ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)
    # fig.tight_layout()
    _paper_dir = globals().get("PAPER_DIR")
    if _paper_dir is not None:
        out = (_paper_dir / "success_rate_vs_delay_all_methods").resolve()
        # fig.savefig(str(out) + ".png", dpi=220)
        fig.savefig(str(out) + ".pdf")
        print("Saved:", str(out) + ".png/.pdf")
    plt.show()

---
## 1. Load Experiment Results

Point to your Section 5 experiment results directory.

In [ ]:
# =============================================================================
# Load Results (systematic, matches the order in
# `scripts/closed_loop_sim/section5_closed_loop_experiments.sh`)
# =============================================================================
# This cell discovers available runs in:
# Current launcher outputs:
#   step3_lambda_sweep_fusions
#   step4_main_results
# Older roots may also contain legacy step1_delay_basics and step2 query sweeps.
# and builds one tidy table (`df_all`) with columns for:
#   - section5_step / delay_sweep / vlm_query_interval_s_sweep
#   - vlm_query_hz_config + vlm_request_hz_effective
#
# We also load a summary table (`df_all_summary`) which includes the new closed-loop
# outcome metrics:
#   - success_rate
#   - mean_route_completion
#   - mean_goal_dist_final_m
#
# It also selects a sensible default `df` + `MAIN_DELAY_S` for the paper-figure sections.

MODE = "auto"  # auto | force_step4 | force_step3

SECTION5_ROOT = resolve_section5_root(
    [
        # Repo-local (portable)
        "logs/section5_closed_loop",
        # "logs/section5_closed_loop_smoke",
    ]
)
print("Section5 root:", SECTION5_ROOT)

# Load everything into one table (row-level results)
try:
    df_all = load_section5_results_table(SECTION5_ROOT, prefer="results")
    df_all_summary = load_section5_results_table(SECTION5_ROOT, prefer="summary")
except Exception:
    print("\n⚠️  Could not load section5 results.")
    print("  Tried root:", SECTION5_ROOT)
    raise

# Split by step (for convenience)
_df_step = {
    int(s): df_all[df_all["section5_step"] == int(s)].copy()
    for s in sorted(df_all["section5_step"].unique())
}
_df_step_summary = {
    int(s): df_all_summary[df_all_summary["section5_step"] == int(s)].copy()
    for s in sorted(df_all_summary["section5_step"].unique())
}

df_step1 = _df_step.get(1)
df_step2 = _df_step.get(2)
df_step3 = _df_step.get(3)
df_step4 = _df_step.get(4)

df_step1_summary = _df_step_summary.get(1)
df_step2_summary = _df_step_summary.get(2)
df_step3_summary = _df_step_summary.get(3)
df_step4_summary = _df_step_summary.get(4)

print("\n📦 Discovered runs (row counts):")
for s in [1, 2, 3, 4]:
    d = _df_step.get(s)
    if d is None:
        print(f"  step{s}: 0")
    else:
        print(f"  step{s}: {len(d)}")

# Choose a default dataset for the existing paper-figure sections.
# Preference: step4 (main results) > step3 (lambda sweep fusions)

df = None
MAIN_DELAY_S = None

if MODE in ("auto", "force_step4") and df_step4 is not None and len(df_step4) > 0:
    df = df_step4
    # Prefer folder-level delay sweep, else row-level delay_s
    cand = (
        df["section5_delay_sweep"].dropna().unique().tolist()
        if "section5_delay_sweep" in df.columns
        else []
    )
    if cand:
        MAIN_DELAY_S = float(sorted(cand)[0])
    elif "delay_s" in df.columns:
        MAIN_DELAY_S = float(sorted(df["delay_s"].dropna().unique().tolist())[0])

if df is None and MODE in ("auto", "force_step3") and df_step3 is not None and len(df_step3) > 0:
    df = df_step3
    cand = (
        df["section5_delay_sweep"].dropna().unique().tolist()
        if "section5_delay_sweep" in df.columns
        else []
    )
    if cand:
        MAIN_DELAY_S = float(sorted(cand)[0])
    elif "delay_s" in df.columns:
        MAIN_DELAY_S = float(sorted(df["delay_s"].dropna().unique().tolist())[0])

if df is None:
    print("\n⚠️  No step3/step4 data found. Run at least one of:")
    print("   bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 1")
    print("   bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 2")
else:
    # Representative run dir for saving figures (keep the existing notebook convention)
    _out_run_dir = (
        Path(str(df["run_dir"].iloc[0])) if "run_dir" in df.columns and len(df) else SECTION5_ROOT
    )
    PAPER_DIR = (_out_run_dir / "paper_figures").resolve()

    print("paper_dir", PAPER_DIR)
    PAPER_DIR.mkdir(parents=True, exist_ok=True)

    # Keep a `run` dict for older cells that expect it
    run = {"run_dir": _out_run_dir}

    print("\n📊 Default dataset for figures:")
    print(
        "  source_step:",
        int(df["section5_step"].iloc[0]) if "section5_step" in df.columns and len(df) else "?",
    )
    print("  rows       :", int(len(df)))
    if "policy" in df.columns:
        print("  policies   :", sorted(df["policy"].unique().tolist()))
    if "delay_s" in df.columns:
        print("  delays     :", sorted(np.unique(df["delay_s"].dropna().to_numpy()).tolist()))
    if "lambda_sim" in df.columns:
        print("  lambdas    :", sorted(np.unique(df["lambda_sim"].dropna().to_numpy()).tolist()))
    if "vlm_query_hz_config" in df.columns:
        print(
            "  vlm_query_hz_config:",
            sorted(np.unique(df["vlm_query_hz_config"].dropna().to_numpy()).tolist()),
        )
    if "vlm_max_inflight" in df.columns:
        print(
            "  vlm_max_inflight:",
            sorted(np.unique(df["vlm_max_inflight"].dropna().to_numpy()).tolist()),
        )

    if MAIN_DELAY_S is not None:
        print("  MAIN_DELAY_S:", MAIN_DELAY_S)

    # Quick check: do we have the new metrics in the loaded tables?
    for col in ["success_rate", "mean_route_completion", "mean_goal_dist_final_m"]:
        if col in df_all_summary.columns:
            continue
        print(f"\n⚠️  Summary column missing: {col}")
        print("  If this is an older run, regenerate it with the updated closed-loop script,")
        print(
            "  or run: python scripts/paper_artifacts/benchmark_closed_loop_score_fusion_real.py "
            "--postprocess-run-dir <run_dir>"
        )

In [ ]:
# =============================================================================
# Figure: X=delay, Y=metrics, lines=policy
# =============================================================================
# This is the most important figure to sanity-check closed-loop behavior on the hard split.
# We plot 3 metrics:
#   - success_rate
#   - mean_route_completion
#   - mean_cte_m (kept for reference)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_df_src = None
if "df_step4_summary" in globals() and df_step4_summary is not None and len(df_step4_summary) > 0:
    _df_src = df_step4_summary.copy()
elif "df_all_summary" in globals() and df_all_summary is not None and len(df_all_summary) > 0:
    _df_src = df_all_summary.copy()

if _df_src is None or len(_df_src) == 0:
    print("No section5 summary data found (run a closed-loop job first).")
else:
    required = ["policy", "delay_s", "success_rate", "mean_route_completion", "mean_cte_m"]
    missing = [c for c in required if c not in _df_src.columns]
    if missing:
        print("Missing columns:", missing)
        print(
            "Tip: regenerate runs with the updated closed-loop script, or postprocess older runs."
        )
    else:
        POLICIES = [
            "planner_oracle",
            "local_only",
            "vlm_hold",
            "vlm_stream",
            "vlm_hold_match",
            "vlm_stream_match",
            "score_fusion",
            "prob_fusion",
            "score_fusion_stream",
            "prob_fusion_stream",
        ]

        dfp = _df_src[_df_src["policy"].isin(POLICIES)].copy()

        # Ensure we have a usable delay column.
        # (Some older loaders keep the folder-inferred delay under section5_delay_sweep;
        # standard is delay_s.)
        if "delay_s" not in dfp.columns:
            if "section5_delay_sweep" in dfp.columns:
                dfp["delay_s"] = dfp["section5_delay_sweep"]
            else:
                raise KeyError("Missing delay column: expected delay_s or section5_delay_sweep")
        dfp["delay_s"] = pd.to_numeric(dfp["delay_s"], errors="coerce")

        # Some runs may include multiple configs per (policy, delay). Pick the best one.
        sort_cols = [
            "success_rate",
            "mean_route_completion",
            (
                "mean_goal_dist_min_m"
                if "mean_goal_dist_min_m" in dfp.columns
                else "mean_goal_dist_final_m"
            ),
            "mean_cte_m",
        ]
        sort_cols = [c for c in sort_cols if c in dfp.columns]
        asc_map = {
            "success_rate": False,
            "mean_route_completion": False,
            "mean_goal_dist_final_m": True,
            "mean_goal_dist_min_m": True,
            "mean_cte_m": True,
        }
        asc = [asc_map.get(c, True) for c in sort_cols]

        # Rank globally by outcome, then take the best row per (policy, delay).
        df_ranked = dfp.sort_values(sort_cols + ["policy", "delay_s"], ascending=asc + [True, True])
        df_best = df_ranked.drop_duplicates(["policy", "delay_s"], keep="first").sort_values(
            ["delay_s", "policy"]
        )

        # Expose for later cells (tables, BEV examples)
        df_best_delay_policy = df_best.copy()

        # Quick sanity: if you see mostly-blank plots, it usually means older runs
        # (before we added success/completion) are being concatenated with NaNs.
        for col in ["success_rate", "mean_route_completion", "mean_goal_dist_final_m"]:
            if col in df_best.columns:
                n_finite = int(np.isfinite(df_best[col].to_numpy(dtype=float)).sum())
                print(f"{col}: finite {n_finite}/{len(df_best)}")

        fig, axes = plt.subplots(1, 4, figsize=(22.0, 4.6), sharex=True)

        plots = [
            ("success_rate", "Success rate", (-0.02, 1.02)),
            ("mean_route_completion", "Mean route completion", (-0.02, 1.02)),
            (
                (
                    "mean_goal_dist_min_m"
                    if "mean_goal_dist_min_m" in df_best.columns
                    else "mean_goal_dist_final_m"
                ),
                "Mean goal dist (m) (min over rollout)",
                None,
            ),
            ("mean_cte_m", "Mean CTE (m)", None),
        ]

        for ax, (col, title, ylim) in zip(axes, plots, strict=False):
            for pol in POLICIES:
                sub = df_best[df_best["policy"] == pol].sort_values("delay_s")
                if len(sub) == 0:
                    continue
                ax.plot(sub["delay_s"], sub[col], marker="o", linewidth=2.0, label=pol)
            ax.set_title(title)
            ax.set_xlabel("Delay (s)")
            ax.grid(True, alpha=0.28)
            if ylim is not None:
                ax.set_ylim(*ylim)

        axes[0].set_ylabel("Metric")
        axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)

        fig.tight_layout()

        if "PAPER_DIR" in globals():
            outp = (PAPER_DIR / "metrics_vs_delay.png").resolve()
            fig.savefig(outp, dpi=220)
            print("Saved:", outp)

        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_df_src = None
if "df_step4_summary" in globals() and df_step4_summary is not None and len(df_step4_summary) > 0:
    _df_src = df_step4_summary.copy()
elif "df_all_summary" in globals() and df_all_summary is not None and len(df_all_summary) > 0:
    _df_src = df_all_summary.copy()

if _df_src is None or len(_df_src) == 0:
    print("No section5 summary data found (run a closed-loop job first).")
else:
    required = ["policy", "delay_s", "success_rate", "mean_route_completion", "mean_cte_m"]
    missing = [c for c in required if c not in _df_src.columns]
    if missing:
        print("Missing columns:", missing)
        print(
            "Tip: regenerate runs with the updated closed-loop script, or postprocess older runs."
        )
    else:
        POLICIES = [
            "planner_oracle",
            "local_only",
            "vlm_hold",
            "vlm_stream",
            "vlm_hold_match",
            "vlm_stream_match",
            "score_fusion",
            "prob_fusion",
            "score_fusion_stream",
            "prob_fusion_stream",
        ]

        dfp = _df_src[_df_src["policy"].isin(POLICIES)].copy()

        # Ensure we have a usable delay column.
        # (Some older loaders keep the folder-inferred delay under section5_delay_sweep;
        # standard is delay_s.)
        if "delay_s" not in dfp.columns:
            if "section5_delay_sweep" in dfp.columns:
                dfp["delay_s"] = dfp["section5_delay_sweep"]
            else:
                raise KeyError("Missing delay column: expected delay_s or section5_delay_sweep")
        dfp["delay_s"] = pd.to_numeric(dfp["delay_s"], errors="coerce")

        # Some runs may include multiple configs per (policy, delay). Pick the best one.
        sort_cols = [
            "success_rate",
            "mean_route_completion",
            (
                "mean_goal_dist_min_m"
                if "mean_goal_dist_min_m" in dfp.columns
                else "mean_goal_dist_final_m"
            ),
            "mean_cte_m",
        ]
        sort_cols = [c for c in sort_cols if c in dfp.columns]
        asc_map = {
            "success_rate": False,
            "mean_route_completion": False,
            "mean_goal_dist_final_m": True,
            "mean_goal_dist_min_m": True,
            "mean_cte_m": True,
        }
        asc = [asc_map.get(c, True) for c in sort_cols]

        # Rank globally by outcome, then take the best row per (policy, delay).
        df_ranked = dfp.sort_values(sort_cols + ["policy", "delay_s"], ascending=asc + [True, True])
        df_best = df_ranked.drop_duplicates(["policy", "delay_s"], keep="first").sort_values(
            ["delay_s", "policy"]
        )

        # Expose for later cells (tables, BEV examples)
        df_best_delay_policy = df_best.copy()

        # Quick sanity: if you see mostly-blank plots, it usually means older runs
        # (before we added success/completion) are being concatenated with NaNs.
        for col in ["success_rate", "mean_route_completion", "mean_goal_dist_final_m"]:
            if col in df_best.columns:
                n_finite = int(np.isfinite(df_best[col].to_numpy(dtype=float)).sum())
                print(f"{col}: finite {n_finite}/{len(df_best)}")

        fig, axes = plt.subplots(1, 4, figsize=(22.0, 4.6), sharex=True)

        plots = [
            ("success_rate", "Success rate", (-0.02, 1.02)),
            ("mean_route_completion", "Mean route completion", (-0.02, 1.02)),
            (
                (
                    "mean_goal_dist_min_m"
                    if "mean_goal_dist_min_m" in df_best.columns
                    else "mean_goal_dist_final_m"
                ),
                "Mean goal dist (m) (min over rollout)",
                None,
            ),
            ("mean_cte_m", "Mean CTE (m)", None),
        ]

        for ax, (col, title, ylim) in zip(axes, plots, strict=False):
            for pol in POLICIES:
                sub = df_best[df_best["policy"] == pol].sort_values("delay_s")
                if len(sub) == 0:
                    continue
                ax.plot(sub["delay_s"], sub[col], marker="o", linewidth=2.0, label=pol)
            ax.set_title(title)
            ax.set_xlabel("Delay (s)")
            ax.grid(True, alpha=0.28)
            if ylim is not None:
                ax.set_ylim(*ylim)

        axes[0].set_ylabel("Metric")
        axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)

        fig.tight_layout()

        if "PAPER_DIR" in globals():
            outp = (PAPER_DIR / "metrics_vs_delay.png").resolve()
            fig.savefig(outp, dpi=220)
            print("Saved:", outp)

        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_df_src = None
if "df_step4_summary" in globals() and df_step4_summary is not None and len(df_step4_summary) > 0:
    _df_src = df_step4_summary.copy()
elif "df_all_summary" in globals() and df_all_summary is not None and len(df_all_summary) > 0:
    _df_src = df_all_summary.copy()

if _df_src is None or len(_df_src) == 0:
    print("No section5 summary data found (run a closed-loop job first).")
else:
    required = ["policy", "delay_s", "success_rate", "mean_route_completion", "mean_cte_m"]
    missing = [c for c in required if c not in _df_src.columns]
    if missing:
        print("Missing columns:", missing)
        print(
            "Tip: regenerate runs with the updated closed-loop script, or postprocess older runs."
        )
    else:
        POLICIES = [
            "planner_oracle",
            "local_only",
            "vlm_hold",
            "vlm_stream",
            "vlm_hold_match",
            "vlm_stream_match",
            "score_fusion",
            "prob_fusion",
            "score_fusion_stream",
            "prob_fusion_stream",
        ]

        dfp = _df_src[_df_src["policy"].isin(POLICIES)].copy()

        # Ensure we have a usable delay column.
        # (Some older loaders keep the folder-inferred delay under section5_delay_sweep;
        # standard is delay_s.)
        if "delay_s" not in dfp.columns:
            if "section5_delay_sweep" in dfp.columns:
                dfp["delay_s"] = dfp["section5_delay_sweep"]
            else:
                raise KeyError("Missing delay column: expected delay_s or section5_delay_sweep")
        dfp["delay_s"] = pd.to_numeric(dfp["delay_s"], errors="coerce")

        # Some runs may include multiple configs per (policy, delay). Pick the best one.
        sort_cols = [
            "success_rate",
            "mean_route_completion",
            (
                "mean_goal_dist_min_m"
                if "mean_goal_dist_min_m" in dfp.columns
                else "mean_goal_dist_final_m"
            ),
            "mean_cte_m",
        ]
        sort_cols = [c for c in sort_cols if c in dfp.columns]
        asc_map = {
            "success_rate": False,
            "mean_route_completion": False,
            "mean_goal_dist_final_m": True,
            "mean_goal_dist_min_m": True,
            "mean_cte_m": True,
        }
        asc = [asc_map.get(c, True) for c in sort_cols]

        # Rank globally by outcome, then take the best row per (policy, delay).
        df_ranked = dfp.sort_values(sort_cols + ["policy", "delay_s"], ascending=asc + [True, True])
        df_best = df_ranked.drop_duplicates(["policy", "delay_s"], keep="first").sort_values(
            ["delay_s", "policy"]
        )

        # Expose for later cells (tables, BEV examples)
        df_best_delay_policy = df_best.copy()

        # Quick sanity: if you see mostly-blank plots, it usually means older runs
        # (before we added success/completion) are being concatenated with NaNs.
        for col in ["success_rate", "mean_route_completion", "mean_goal_dist_final_m"]:
            if col in df_best.columns:
                n_finite = int(np.isfinite(df_best[col].to_numpy(dtype=float)).sum())
                print(f"{col}: finite {n_finite}/{len(df_best)}")

        fig, axes = plt.subplots(1, 4, figsize=(22.0, 4.6), sharex=True)

        plots = [
            ("success_rate", "Success rate", (-0.02, 1.02)),
            ("mean_route_completion", "Mean route completion", (-0.02, 1.02)),
            (
                (
                    "mean_goal_dist_min_m"
                    if "mean_goal_dist_min_m" in df_best.columns
                    else "mean_goal_dist_final_m"
                ),
                "Mean goal dist (m) (min over rollout)",
                None,
            ),
            ("mean_cte_m", "Mean CTE (m)", None),
        ]

        for ax, (col, title, ylim) in zip(axes, plots, strict=False):
            for pol in POLICIES:
                sub = df_best[df_best["policy"] == pol].sort_values("delay_s")
                if len(sub) == 0:
                    continue
                ax.plot(sub["delay_s"], sub[col], marker="o", linewidth=2.0, label=pol)
            ax.set_title(title)
            ax.set_xlabel("Delay (s)")
            ax.grid(True, alpha=0.28)
            if ylim is not None:
                ax.set_ylim(*ylim)

        axes[0].set_ylabel("Metric")
        axes[-1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)

        fig.tight_layout()

        if "PAPER_DIR" in globals():
            outp = (PAPER_DIR / "metrics_vs_delay.png").resolve()
            fig.savefig(outp, dpi=220)
            print("Saved:", outp)

        plt.show()

In [ ]:
# =============================================================================
# Table view: rows=policy, columns=delay
# =============================================================================

import numpy as np
import pandas as pd

if (
    "df_best_delay_policy" not in globals()
    or df_best_delay_policy is None
    or len(df_best_delay_policy) == 0
):
    print("Run the previous plotting cell first (it defines df_best_delay_policy).")
else:
    dfb = df_best_delay_policy.copy()

    def pivot_metric(col: str, fmt: str = "{:.3f}") -> pd.DataFrame:
        if col not in dfb.columns:
            return pd.DataFrame()
        pt = dfb.pivot(index="policy", columns="delay_s", values=col)
        pt = pt.reindex(POLICIES)
        pt = pt.sort_index(axis=1)

        # Pretty formatting for display.
        # NOTE: pandas>=3 removed DataFrame.applymap; use a numpy-based formatter instead.
        num = pt.apply(pd.to_numeric, errors="coerce")
        a = num.to_numpy(dtype=float, copy=False)
        mask = np.isfinite(a)
        out = np.full(a.shape, "", dtype=object)
        if mask.any():
            out[mask] = np.vectorize(lambda v: fmt.format(float(v)))(a[mask])
        return pd.DataFrame(out, index=num.index, columns=num.columns)

    print("\nSuccess rate (rows=policy, cols=delay_s)")
    display(pivot_metric("success_rate"))

    print("\nMean route completion")
    display(pivot_metric("mean_route_completion"))

    if "mean_goal_dist_min_m" in dfb.columns:
        print("\nMean goal dist min (m)  (lower is better; min over rollout)")
        display(pivot_metric("mean_goal_dist_min_m", fmt="{:.2f}"))
    elif "mean_goal_dist_final_m" in dfb.columns:
        print("\nMean goal dist final (m)  (lower is better)")
        display(pivot_metric("mean_goal_dist_final_m", fmt="{:.2f}"))

    print("\nMean CTE (m)  (lower is better)")
    display(pivot_metric("mean_cte_m", fmt="{:.3f}"))

In [ ]:
# =============================================================================
# BEV examples: visualize how the robot walks (single episode prefix)
# =============================================================================
# This cell shows pre-rendered BEV PNGs written by:
#   python scripts/paper_artifacts/benchmark_closed_loop_score_fusion_real.py --write-examples
# or via:
#   WRITE_EXAMPLES=1 bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 1
#
# It will display ALL methods for ONE episode prefix, including local planner + oracle.

from pathlib import Path

from IPython.display import Image, display

root = Path(str(SECTION5_ROOT)).resolve()
base = root / "step4_main_results" / "delay_2.0"

PREFIX = "task01_2025-08-26__C10247__cbd19ce4-a00f-421f-9f80-2012f80c06e6__1756225517006884352_0_"

# Where each policy's PNG usually lives.
# - baselines run_dir contains multiple policies (local_only, planner_oracle, vlm_*, *_match)
# - fusion folders contain their single policy
folders = [
    "baselines",
    "score_fusion",
    "prob_fusion",
    "score_fusion_stream",
    "prob_fusion_stream",
]

policy_order = [
    "planner_oracle",
    "local_only",
    "vlm_hold",
    "vlm_stream",
    "vlm_hold_match",
    "vlm_stream_match",
    "score_fusion",
    "prob_fusion",
    "score_fusion_stream",
    "prob_fusion_stream",
]

# Pick newest run_dir per folder, then gather matching PNGs.
policy_to_png: dict[str, Path] = {}
folder_to_run: dict[str, Path] = {}

for folder in folders:
    run_parent = base / folder
    runs = sorted([p for p in run_parent.glob("closed_loop_fusion_*") if p.is_dir()])
    if not runs:
        continue
    run_dir = runs[-1]
    folder_to_run[folder] = run_dir

    ex_dir = run_dir / "examples_bev"
    if not ex_dir.exists():
        continue
    for p in ex_dir.glob(f"{PREFIX}*.png"):
        name = p.name
        if not name.startswith(PREFIX) or not name.endswith(".png"):
            continue
        pol = name[len(PREFIX) : -len(".png")]
        policy_to_png[str(pol)] = p

if not policy_to_png:
    print("No BEV PNGs found for prefix under:", base)
else:
    print("PREFIX:", PREFIX)
    print("\nUsing newest runs:")
    for folder in folders:
        if folder in folder_to_run:
            print(f"- {folder}: {folder_to_run[folder]}")

    missing = [p for p in policy_order if p not in policy_to_png]
    if missing:
        print("\nMissing policies (no PNG found):")
        for p in missing:
            print("-", p)

    print("\nShowing BEVs:")
    for pol in policy_order:
        p = policy_to_png.get(pol)
        if p is None:
            continue
        display(Image(filename=str(p)))
        print(p.name)

---
## 2. Step 1: Delay basics (vlm_hold vs vlm_stream)

This is a **legacy delay-basics view** for older result roots.

It loads `step1_delay_basics` when that historical folder is present. The current launcher writes delay sweeps to `step4_main_results`; use the current delay-sweep section below for new runs.
- `local_only`
- `vlm_hold`
- `vlm_stream`


In [ ]:
# =============================================================================
# Delay sweep: policy performance vs delay (step4_main_results)
# =============================================================================
# This matches the delay sweep loop (step 2) in
# `scripts/closed_loop_sim/section5_closed_loop_experiments.sh`.

if df_step4 is None or len(df_step4) == 0:
    print("No step4_main_results data found.")
    print("Run: bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 2")
else:
    d = df_step4.copy()

    # Use folder key when available.
    if "section5_delay_sweep" in d.columns:
        d["delay_sweep"] = d["section5_delay_sweep"]
    else:
        d["delay_sweep"] = d.get("delay_s", np.nan)

    METRIC = "mean_cte_m"
    if METRIC not in d.columns:
        raise KeyError(f"Expected {METRIC} in step4 results. Columns: {sorted(d.columns)}")

    # Aggregate over task windows. If multiple lambdas exist, select the best lambda
    # per (delay, policy)
    # so the curve reflects the best achieved configuration at each delay.
    if "lambda_sim" in d.columns:
        agg = (
            d.groupby(["delay_sweep", "policy", "lambda_sim"], as_index=False)
            .agg(mean=(METRIC, "mean"), std=(METRIC, "std"), n=(METRIC, "count"))
            .sort_values(["delay_sweep", "policy", "lambda_sim"])
        )
        best = (
            agg.loc[agg.groupby(["delay_sweep", "policy"])["mean"].idxmin()]
            .copy()
            .reset_index(drop=True)
        )
    else:
        best = (
            d.groupby(["delay_sweep", "policy"], as_index=False)
            .agg(mean=(METRIC, "mean"), std=(METRIC, "std"), n=(METRIC, "count"))
            .sort_values(["delay_sweep", "policy"])
        )
        best["lambda_sim"] = np.nan

    display(best.sort_values(["delay_sweep", "policy"]).head(40))

    # Stable plotting order (only plot policies present)
    PREFERRED_ORDER = [
        "local_only",
        "vlm_hold",
        "vlm_stream",
        # "vlm_hold_match",
        # "vlm_stream_match",
        "score_fusion",
        "prob_fusion",
        "score_fusion_stream",
        "prob_fusion_stream",
    ]
    present = best["policy"].dropna().unique().tolist()
    # policies = [p for p in PREFERRED_ORDER if p in present] + [
    #     p for p in sorted(present) if p not in PREFERRED_ORDER
    # ]

    policies = [p for p in PREFERRED_ORDER if p in present]

    fig, ax = plt.subplots(figsize=(10, 10))
    for policy in policies:
        pdp = best[best["policy"] == policy].sort_values("delay_sweep")
        if len(pdp) == 0:
            continue
        ax.plot(
            pdp["delay_sweep"],
            pdp["mean"],
            marker=POLICY_MARKERS.get(policy, "o"),
            color=POLICY_COLORS.get(policy, "#333"),
            label=POLICY_LABELS.get(policy, policy),
            linewidth=2,
        )

    ax.set_xlabel("VLM delay (s)")
    ax.set_ylabel("Mean CTE (m)")
    ax.set_title("Delay sweep: mean CTE vs VLM delay (step4 main results)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    if "PAPER_DIR" in globals():
        out = PAPER_DIR / "fig_delay_sweep_all_policies"
        fig.savefig(str(out) + ".pdf", bbox_inches="tight")
        fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
        print(f"Saved: {out}.pdf/.png")

    plt.show()

---
## 3. Step 2: VLM streaming frequency sweep (vlm_stream*)

This is a **legacy query-interval view** for older result roots.

It loads `step2_vlm_query_interval_sweep` (or `step2_vlm_query_rate_sweep`) only when those historical folders are present. The current two-step launcher does not generate a query-interval sweep.

- Supports folder keys:
  - preferred: `delay_<delay>/interval_s_<interval_s>`
  - legacy: `delay_<delay>/hz_<hz>`
- For fusion variants, selects **best λ per (delay, interval, policy)** to isolate the effect of streaming cadence


In [ ]:
# =============================================================================
# Step 2 plots: mean CTE vs VLM query interval (seconds/query)
# =============================================================================
# Terminology:
# - query rate (Hz)      = queries / second
# - query interval (s)   = seconds / query = 1 / Hz
# This cell plots interval (seconds) to avoid unit confusion.

if df_step2 is None or len(df_step2) == 0:
    print("No step2_vlm_query_interval_sweep data found (or legacy step2_vlm_query_rate_sweep).")
    print(
        "The current launcher does not generate this legacy query-interval sweep; "
        "skip this section or point SECTION5_ROOT at historical results."
    )
else:
    df_q = df_step2.copy()

    # Normalize sweep keys
    df_q["delay_sweep"] = df_q.get("section5_delay_sweep", np.nan)
    df_q["vlm_query_hz_sweep"] = df_q.get("section5_vlm_query_hz_sweep", np.nan)

    # Convert to interval (seconds/query)
    with np.errstate(divide="ignore", invalid="ignore"):
        df_q["vlm_query_interval_s_sweep"] = np.where(
            np.isfinite(df_q["vlm_query_hz_sweep"]) & (df_q["vlm_query_hz_sweep"] > 0),
            1.0 / df_q["vlm_query_hz_sweep"],
            np.nan,
        )

    STREAM_POLICIES = [
        "vlm_stream",
        "vlm_stream_match",
    ]
    df_q = df_q[df_q.get("policy").isin(STREAM_POLICIES)].copy()

    METRIC = "mean_cte_m"
    if METRIC not in df_q.columns:
        raise KeyError(
            f"Expected column '{METRIC}' in step2 results. Columns: {sorted(df_q.columns)}"
        )

    # Summarize: best λ per (delay, hz, policy) for fusion variants.
    def _summarize_group(g: pd.DataFrame) -> pd.Series:
        delay_val, interval_s, policy = g.name
        is_fusion = "fusion" in str(policy)

        gg = g
        best_lam = np.nan
        if is_fusion and "lambda_sim" in g.columns and g["lambda_sim"].nunique() > 1:
            by_lam = g.groupby("lambda_sim")[METRIC].mean()
            best_lam = float(by_lam.idxmin())
            gg = g[g["lambda_sim"] == best_lam]
        elif "lambda_sim" in g.columns and len(g):
            best_lam = float(g["lambda_sim"].iloc[0])

        return pd.Series(
            {
                "mean": float(gg[METRIC].mean()),
                "std": float(gg[METRIC].std()),
                "best_lambda": float(best_lam) if np.isfinite(best_lam) else np.nan,
                "n_rows": int(len(gg)),
                "mean_vlm_request_hz_effective": float(
                    np.nanmean(gg.get("vlm_request_hz_effective", np.nan))
                ),
            }
        )

    agg_q = (
        df_q.groupby(["delay_sweep", "vlm_query_interval_s_sweep", "policy"], group_keys=False)
        .apply(_summarize_group)
        .reset_index()
        .sort_values(["delay_sweep", "policy", "vlm_query_interval_s_sweep"])
    )

    display(agg_q.head(30))

    # Plot: one panel per delay (ordered by sweep values)
    for delay_val in sorted([x for x in agg_q["delay_sweep"].unique().tolist() if np.isfinite(x)]):
        fig, ax = plt.subplots(figsize=(9, 4.8))
        dd = agg_q[agg_q["delay_sweep"] == delay_val]

        for policy in STREAM_POLICIES:
            pdp = dd[dd["policy"] == policy].sort_values("vlm_query_interval_s_sweep")
            if len(pdp) == 0:
                continue

            # Plot mean ONLY, no error bar
            ax.plot(
                pdp["vlm_query_interval_s_sweep"],
                pdp["mean"],
                marker=POLICY_MARKERS.get(policy, "o"),
                color=POLICY_COLORS.get(policy, "#333"),
                label=POLICY_LABELS.get(policy, policy),
                linewidth=2,
            )

        ax.set_xlabel("VLM query interval (s)")
        ax.set_ylabel("Mean CTE (m)")
        ax.set_title(f"Step 2: query-interval sweep (delay = {delay_val:g}s)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
        plt.tight_layout()

        if "PAPER_DIR" in globals():
            out = PAPER_DIR / f"fig_step2_query_interval_delay_{delay_val:g}"
            fig.savefig(str(out) + ".pdf", bbox_inches="tight")
            fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
            print(f"Saved: {out}.pdf/.png")

        plt.show()

    # Sanity: effective interval vs configured interval
    if "mean_vlm_request_hz_effective" in agg_q.columns:
        with np.errstate(divide="ignore", invalid="ignore"):
            eff_interval = np.where(
                np.isfinite(agg_q["mean_vlm_request_hz_effective"])
                & (agg_q["mean_vlm_request_hz_effective"] > 0),
                1.0 / agg_q["mean_vlm_request_hz_effective"],
                np.nan,
            )

        fig, ax = plt.subplots(figsize=(7.5, 4.2))
        ax.scatter(
            agg_q["vlm_query_interval_s_sweep"],
            eff_interval,
            s=40,
            alpha=0.8,
            color="#333333",
        )
        lo = float(np.nanmin(agg_q["vlm_query_interval_s_sweep"]))
        hi = float(np.nanmax(agg_q["vlm_query_interval_s_sweep"]))
        ax.plot([lo, hi], [lo, hi], "--", color="#999999", linewidth=1.5, label="y = x")
        ax.set_xlabel("Configured query interval (s) [folder]")
        ax.set_ylabel("Effective query interval (s) [time / vlm_requests]")
        ax.set_title("Step 2 sanity: effective query interval")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
        plt.tight_layout()
        plt.show()

In [ ]:
# =============================================================================
# Step 2 summary: best query interval per (delay, policy)
# =============================================================================
# This is the "knob selection" view for STEP 2:
# - picks best interval (s/query) by minimizing mean CTE
# - for fusion variants, Step 2 above already selected best λ per (delay, interval, policy)

if "agg_q" not in globals() or agg_q is None or len(agg_q) == 0:
    print("Step 2 summary requires `agg_q` from the Step 2 plot cell above.")
    print("Run the Step 2 cell first.")
else:
    best_interval = (
        agg_q.loc[agg_q.groupby(["delay_sweep", "policy"])["mean"].idxmin()]
        .copy()
        .sort_values(["delay_sweep", "policy"])
        .reset_index(drop=True)
    )

    print("\n✅ Step 2: best interval per (delay, policy) [min mean CTE]")
    display(
        best_interval[
            [
                "delay_sweep",
                "policy",
                "vlm_query_interval_s_sweep",
                "mean",
                "std",
                "best_lambda",
                "mean_vlm_request_hz_effective",
            ]
        ]
    )

    # Also compute a "global" best interval per delay (average across stream policies)
    by_interval = (
        agg_q.groupby(["delay_sweep", "vlm_query_interval_s_sweep"], as_index=False)
        .agg(mean_over_policies=("mean", "mean"))
        .sort_values(["delay_sweep", "vlm_query_interval_s_sweep"])
    )
    best_interval_global = (
        by_interval.loc[by_interval.groupby(["delay_sweep"])["mean_over_policies"].idxmin()]
        .sort_values("delay_sweep")
        .reset_index(drop=True)
    )

    print("\n✅ Step 2: best interval per delay (avg over stream policies)")
    display(best_interval_global)

    # Plot: best interval vs delay
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for policy in [p for p in STREAM_POLICIES if p in best_interval["policy"].unique().tolist()]:
        pdp = best_interval[best_interval["policy"] == policy].sort_values("delay_sweep")
        if len(pdp) == 0:
            continue
        ax.plot(
            pdp["delay_sweep"],
            pdp["vlm_query_interval_s_sweep"],
            marker=POLICY_MARKERS.get(policy, "o"),
            color=POLICY_COLORS.get(policy, "#333"),
            label=POLICY_LABELS.get(policy, policy),
            linewidth=2,
        )

    ax.plot(
        best_interval_global["delay_sweep"],
        best_interval_global["vlm_query_interval_s_sweep"],
        "--",
        color="#333333",
        linewidth=2,
        label="Best interval (avg over policies)",
    )

    ax.set_xlabel("VLM delay (s)")
    ax.set_ylabel("Best VLM query interval (s/query)")
    ax.set_title("Step 2: best streaming interval vs delay")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()

    if "PAPER_DIR" in globals():
        out = PAPER_DIR / "fig_step2_best_interval_vs_delay"
        fig.savefig(str(out) + ".pdf", bbox_inches="tight")
        fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
        print(f"Saved: {out}.pdf/.png")

    plt.show()

    # Convenience: if you're picking a single interval for STEP 4,
    # the most common choice is to use the MAIN_DELAY_S row (if present).
    if "MAIN_DELAY_S" in globals() and MAIN_DELAY_S is not None:
        dd = best_interval_global[best_interval_global["delay_sweep"] == float(MAIN_DELAY_S)]
        if len(dd):
            print(
                "\nSuggested knob for STEP 4 (from Step 2):\n"
                f"  BEST_VLM_QUERY_INTERVAL_S={float(dd['vlm_query_interval_s_sweep'].iloc[0]):g}"
            )

---
## 4. Step 3: λ sweep (fusion methods)

This section corresponds to the λ-sweep step (`bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 1`).

It plots **mean CTE vs λ** for:
- `score_fusion`, `prob_fusion`

It also prints the **best λ** (min mean CTE) per policy, which you can copy into `BEST_LAMBDA_*` env vars for the delay-sweep step.

In [ ]:
# =============================================================================
# Step 3 plots: mean CTE vs λ (lambda_sim)
# =============================================================================

STEP3_POLICIES = [
    "score_fusion",
    "prob_fusion",
    "score_fusion_stream",
    "prob_fusion_stream",
]

if df_step3 is None or len(df_step3) == 0:
    print("No step3_lambda_sweep_fusions data found.")
    print("Run: bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 1")
else:
    d3 = df_step3.copy()

    d3["delay_sweep"] = d3.get("section5_delay_sweep", np.nan)
    d3["vlm_query_interval_s_sweep"] = d3.get("section5_vlm_query_interval_s_sweep", np.nan)

    # Fall back to injected helper column when present
    if "vlm_streaming_interval_s" in d3.columns:
        d3["vlm_query_interval_s_sweep"] = d3["vlm_query_interval_s_sweep"].fillna(
            d3["vlm_streaming_interval_s"]
        )

    d3 = d3[d3.get("policy").isin(STEP3_POLICIES)].copy()

    METRIC = "mean_cte_m"
    if METRIC not in d3.columns:
        raise KeyError(
            f"Expected column '{METRIC}' in step3 results. Columns: {sorted(d3.columns)}"
        )
    if "lambda_sim" not in d3.columns:
        raise KeyError(
            f"Expected column 'lambda_sim' in step3 results. Columns: {sorted(d3.columns)}"
        )

    agg3 = (
        d3.groupby(
            ["delay_sweep", "vlm_query_interval_s_sweep", "policy", "lambda_sim"], as_index=False
        )
        .agg(
            mean=(METRIC, "mean"),
            std=(METRIC, "std"),
            n=(METRIC, "count"),
            switches=("chosen_switches", "mean")
            if "chosen_switches" in d3.columns
            else (METRIC, "count"),
            mean_vlm_request_hz_effective=("vlm_request_hz_effective", "mean")
            if "vlm_request_hz_effective" in d3.columns
            else (METRIC, "count"),
        )
        .sort_values(["delay_sweep", "vlm_query_interval_s_sweep", "policy", "lambda_sim"])
    )

    display(agg3.head(30))

    best3 = (
        agg3.loc[
            agg3.groupby(["delay_sweep", "vlm_query_interval_s_sweep", "policy"])["mean"].idxmin()
        ]
        .copy()
        .sort_values(["delay_sweep", "vlm_query_interval_s_sweep", "policy"])
        .reset_index(drop=True)
    )

    print("\n✅ Step 3: best λ per (delay, interval, policy) [min mean CTE]")
    display(
        best3[["delay_sweep", "vlm_query_interval_s_sweep", "policy", "lambda_sim", "mean", "std"]]
    )

    # Suggested env vars for STEP 4
    # (use MAIN_DELAY_S if present; otherwise use the first delay/interval)
    target_delay = (
        float(MAIN_DELAY_S) if "MAIN_DELAY_S" in globals() and MAIN_DELAY_S is not None else None
    )
    if target_delay is None:
        target_delay = float(
            sorted([x for x in best3["delay_sweep"].unique().tolist() if np.isfinite(x)])[0]
        )

    # Pick a representative interval (step3 usually runs one interval)
    target_interval = (
        best3[best3["delay_sweep"] == target_delay]["vlm_query_interval_s_sweep"]
        .dropna()
        .unique()
        .tolist()
    )
    target_interval = float(sorted(target_interval)[0]) if target_interval else np.nan

    bb = best3[
        (best3["delay_sweep"] == target_delay)
        & (best3["vlm_query_interval_s_sweep"] == target_interval)
    ]
    if len(bb):

        def _get_lam(policy: str) -> float | None:
            row = bb[bb["policy"] == policy]
            if len(row) == 0:
                return None
            v = row["lambda_sim"].iloc[0]
            return float(v) if np.isfinite(v) else None

        print("\nSuggested knobs for STEP 4 (from Step 3):")
        print(f"  # (delay={target_delay:g}s, interval={target_interval:g}s)")

        for p in STEP3_POLICIES:
            v = _get_lam(p)
            if v is not None:
                print(f"  BEST_LAMBDA_{p}={v:g}")

    # Plot: one panel per (delay, interval)
    combos = (
        agg3[["delay_sweep", "vlm_query_interval_s_sweep"]]
        .dropna()
        .drop_duplicates()
        .sort_values(["delay_sweep", "vlm_query_interval_s_sweep"])
    )

    for delay_val, interval_val in combos.itertuples(index=False):
        fig, ax = plt.subplots(figsize=(9, 4.8))
        dd = agg3[
            (agg3["delay_sweep"] == delay_val)
            & (agg3["vlm_query_interval_s_sweep"] == interval_val)
        ]

        for policy in STEP3_POLICIES:
            pdp = dd[dd["policy"] == policy].sort_values("lambda_sim")
            if len(pdp) == 0:
                continue
            ax.plot(
                pdp["lambda_sim"],
                pdp["mean"],
                marker=POLICY_MARKERS.get(policy, "o"),
                color=POLICY_COLORS.get(policy, "#333"),
                label=POLICY_LABELS.get(policy, policy),
                linewidth=2,
            )

            # Highlight best λ
            bl = best3[
                (best3["delay_sweep"] == delay_val)
                & (best3["vlm_query_interval_s_sweep"] == interval_val)
                & (best3["policy"] == policy)
            ]
            if len(bl):
                ax.scatter(
                    [float(bl["lambda_sim"].iloc[0])],
                    [float(bl["mean"].iloc[0])],
                    s=90,
                    facecolors="none",
                    edgecolors=POLICY_COLORS.get(policy, "#333"),
                    linewidths=2,
                    zorder=5,
                )

        ax.set_xlabel("λ (lambda_sim)")
        ax.set_ylabel("Mean CTE (m)")
        ax.set_title(f"Step 3: λ sweep (delay = {delay_val:g}s, interval = {interval_val:g}s)")
        ax.grid(True, alpha=0.3)
        ax.legend(loc="best")
        plt.tight_layout()

        if "PAPER_DIR" in globals():
            out = (
                PAPER_DIR
                / f"fig_step3_lambda_sweep_delay_{delay_val:g}_interval_s_{interval_val:g}"
            )
            fig.savefig(str(out) + ".pdf", bbox_inches="tight")
            fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
            print(f"Saved: {out}.pdf/.png")

        plt.show()

---
## 5. Step 4: Main results (fixed best knobs)

This section corresponds to the delay-sweep step (`bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 2`).

It produces a **policy comparison** plot at each delay using the configs you chose from Step 2 (best interval) and Step 3 (best λ).

In [ ]:
# =============================================================================
# Step 4 plots: main results (policy comparison)
# =============================================================================

if df_step4 is None or len(df_step4) == 0:
    print("No step4_main_results data found.")
    print("Run: bash scripts/closed_loop_sim/section5_closed_loop_experiments.sh 2")
    print("(step 2 requires BEST_* env vars; see the launcher script output.)")
else:
    d4 = df_step4.copy()
    d4["delay_sweep"] = d4.get("section5_delay_sweep", np.nan)

    METRIC = "mean_cte_m"
    if METRIC not in d4.columns:
        raise KeyError(
            f"Expected column '{METRIC}' in step4 results. Columns: {sorted(d4.columns)}"
        )

    group_cols = ["delay_sweep", "policy"]
    has_lambda = "lambda_sim" in d4.columns and d4["lambda_sim"].nunique() > 1
    if has_lambda:
        group_cols.append("lambda_sim")

    agg4 = d4.groupby(group_cols, as_index=False).agg(
        mean=(METRIC, "mean"),
        std=(METRIC, "std"),
        n=(METRIC, "count"),
        switches=("chosen_switches", "mean")
        if "chosen_switches" in d4.columns
        else (METRIC, "count"),
    )

    # If multiple lambdas exist in step4 (unexpected), pick best.
    if has_lambda:
        best4 = (
            agg4.loc[agg4.groupby(["delay_sweep", "policy"])["mean"].idxmin()]
            .copy()
            .reset_index(drop=True)
        )
    else:
        best4 = agg4.copy()

    # Order policies
    best4["policy_order"] = best4["policy"].map({p: i for i, p in enumerate(POLICY_ORDER)})
    best4 = best4.sort_values(["delay_sweep", "policy_order"], na_position="last")

    print("\n✅ Step 4: aggregated main results")
    cols = ["delay_sweep", "policy", "mean", "std"]
    if "lambda_sim" in best4.columns:
        cols.insert(3, "lambda_sim")
    if "switches" in best4.columns:
        cols.append("switches")
    display(best4[cols])

    # Bar chart per delay
    for delay_val in sorted([x for x in best4["delay_sweep"].unique().tolist() if np.isfinite(x)]):
        dd = best4[best4["delay_sweep"] == delay_val].copy()
        dd = dd[dd["policy"].isin(POLICY_ORDER)].copy()
        dd["policy_order"] = dd["policy"].map({p: i for i, p in enumerate(POLICY_ORDER)})
        dd = dd.sort_values("policy_order")

        fig, ax = plt.subplots(figsize=(9.5, 4.8))
        x = np.arange(len(dd))
        bars = ax.bar(
            x,
            dd["mean"].values,
            color=[POLICY_COLORS.get(p, "#888888") for p in dd["policy"]],
            edgecolor="#333333",
            linewidth=0.6,
        )

        if PLOT_ERROR_BARS and "std" in dd.columns:
            ax.errorbar(
                x,
                dd["mean"].values,
                yerr=dd["std"].values,
                fmt="none",
                ecolor="#333333",
                elinewidth=1,
                capsize=3,
                zorder=10,
            )

        ax.set_xticks(x)
        ax.set_xticklabels([POLICY_LABELS.get(p, p) for p in dd["policy"]], rotation=20, ha="right")
        ax.set_ylabel("Mean CTE (m)")
        ax.set_title(f"Step 4: main results (delay = {delay_val:g}s)")
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()

        if "PAPER_DIR" in globals():
            out = PAPER_DIR / f"fig_step4_main_results_delay_{delay_val:g}"
            fig.savefig(str(out) + ".pdf", bbox_inches="tight")
            fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
            print(f"Saved: {out}.pdf/.png")

        plt.show()

    # Scatter: CTE vs switches (accuracy vs smoothness)
    if "switches" in best4.columns:
        for delay_val in sorted(
            [x for x in best4["delay_sweep"].unique().tolist() if np.isfinite(x)]
        ):
            dd = best4[best4["delay_sweep"] == delay_val].copy()
            fig, ax = plt.subplots(figsize=(7.5, 5.2))
            for policy in dd["policy"].unique().tolist():
                row = dd[dd["policy"] == policy]
                if len(row) == 0:
                    continue
                ax.scatter(
                    row["switches"].values,
                    row["mean"].values,
                    s=80,
                    marker=POLICY_MARKERS.get(policy, "o"),
                    color=POLICY_COLORS.get(policy, "#333"),
                    edgecolors="#222222",
                    linewidths=0.5,
                    label=POLICY_LABELS.get(policy, policy),
                    alpha=0.85,
                )

            ax.set_xlabel("Trajectory switches / episode")
            ax.set_ylabel("Mean CTE (m)")
            ax.set_title(f"Step 4: accuracy vs smoothness (delay = {delay_val:g}s)")
            ax.grid(True, alpha=0.3)
            ax.legend(loc="best")
            plt.tight_layout()

            if "PAPER_DIR" in globals():
                out = PAPER_DIR / f"fig_step4_cte_vs_switches_delay_{delay_val:g}"
                fig.savefig(str(out) + ".pdf", bbox_inches="tight")
                fig.savefig(str(out) + ".png", dpi=300, bbox_inches="tight")
                print(f"Saved: {out}.pdf/.png")

            plt.show()